# เตรียมเสียงร่วม: คน + Wayu → WAV mono 16 kHz

ขั้นนี้อ่านรายการสองคลาสจาก pilot แล้วสร้างไฟล์ใหม่ **Train 160 / Dev 40** ให้เป็นรูปแบบเดียวกัน ใช้ได้ร่วมกันสำหรับ AASIST และ RawNet2 ไม่คัดลอก pipeline ให้โมเดลละชุด

เลือก **Kernel → .venv (Python 3.12)** ไม่ใช่ `.venv-wayu` และรันจากบนลงล่าง ไม่ต้องโหลด TTS หรือรัน notebook สร้างเสียงใหม่

- WAV float32 (FLOAT), mono, 16 kHz ทั้งสองคลาส
- เก็บความยาวเต็ม ไม่ตัดเหลือ 4.04 วินาที ไม่วน/เติมเสียงในขั้นนี้
- ไม่ normalize ระดับเสียง ไม่ clip/แปลง PCM16 ไม่ตัดช่วงเงียบ และไม่เพิ่ม noise/telephone
- ไม่แก้ raw Common Voice, เสียง Wayu native, manifest หรือ reviews เดิม ไม่ train
- การเปลี่ยนรูปแบบไฟล์ **ไม่ได้ลบผลของ MP3 เดิมหรือทำให้แบนด์วิดท์/คุณภาพต้นทางเท่ากัน** จึงยังไม่อ้างว่าควบคุมทุกความต่างระหว่างคลาส

โค้ดส่วนกลางอยู่ `src/thai_spoof/cvtts/audio.py` และ `artifacts.py` มี tests สำหรับ downmix/resample/ช่วงค่า/duration และไม่เขียนทับไฟล์ ไม่ใช้ helper inference ที่ปรับ peak เกิน 1 โดยอัตโนมัติ

In [ ]:
from pathlib import Path
import sys, io, json, hashlib, time, platform
import importlib.metadata as metadata
import pandas as pd
import numpy as np
import soundfile as sf
from IPython.display import display, Audio, Markdown

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("เปิด notebook ภายใน project ก่อน")
    PROJECT_ROOT = PROJECT_ROOT.parent
assert Path(sys.executable).resolve() == (PROJECT_ROOT / ".venv/Scripts/python.exe").resolve(), "เลือก .venv หลัก"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from thai_spoof.cvtts.audio import (
    CANONICAL_POLICY, TARGET_SAMPLE_RATE, read_and_resample, encode_float32_wav,
)
from thai_spoof.cvtts.artifacts import write_immutable_artifacts

PILOT_DIR = PROJECT_ROOT / "data/processed/cvtts/pilot_v1"
MANIFEST_DIR = PILOT_DIR / "manifests"
INPUT_REPORT_PATH = PILOT_DIR / "qc/tts_experiments/wayu_pilot_train_dev_v1.json"
input_report = json.loads(INPUT_REPORT_PATH.read_text(encoding="utf-8"))
EXPERIMENT = "wayu_pilot_clean16k_v1"
source_tables, source_manifest_hashes = {}, {}
for split, size in [("train", 160), ("dev", 40)]:
    path = MANIFEST_DIR / f"wayu_pilot_{split}_native.csv"
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == input_report["manifest_sha256"][path.name]
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    assert len(frame) == size and frame["split"].eq(split).all()
    assert frame["label"].value_counts().to_dict() == {"bonafide": size // 2, "spoof": size // 2}
    assert frame["sample_id"].is_unique and frame["audio_file_sha256"].is_unique
    assert frame["source_sample_id"].ne("").all() and frame["source_text_key"].ne("").all()
    for column in ["source_sample_id", "source_text_key"]:
        for _, group in frame.groupby(column, sort=False):
            assert len(group) == 2 and set(group["label"]) == {"bonafide", "spoof"}
    source_tables[split] = frame
    source_manifest_hashes[path.name] = digest
for column in ["sample_id", "source_sample_id", "source_text_key", "audio_file_sha256"]:
    assert set(source_tables["train"][column]).isdisjoint(set(source_tables["dev"][column])), column

print("รายการเดิมครบ: Train 160 / Dev 40; ไม่มีข้อมูลร่วมกันข้าม split")
display(pd.concat(source_tables.values(), ignore_index=True).groupby(["split", "label"]).size().rename("clips").to_frame())

## 1. อ่านเสียงเต็มและเตรียมไฟล์ใหม่

อ่านเสียงต้นทางและเช็ก hash ก่อนแปลง stereo → ค่าเฉลี่ยช่องเสียง; resample ด้วย `scipy.signal.resample_poly` ถ้า sample rate ต่าง ไม่ปรับ gain หรือทำให้ความยาวเท่ากัน

WAV float32 เก็บค่า waveform โดยไม่บีบให้เป็น PCM16 ตรวจอ่านกลับตรงกันและความยาวเปลี่ยนไม่เกิน 1 sample ของ 16 kHz เตรียมทั้งหมดใน memory แล้วตรวจ split/duplicate ก่อนเขียน

In [ ]:
prepared_files, canonical_rows = {}, []
started = time.perf_counter()
index = 0
for split in ["train", "dev"]:
    for _, row in source_tables[split].iterrows():
        index += 1
        label = row["label"]
        assert label in {"bonafide", "spoof"}
        source_path = (PROJECT_ROOT / row["audio_path"]).resolve()
        expected_source_root = (
            PROJECT_ROOT / "data/raw/common_voice"
            if label == "bonafide" else PILOT_DIR / "native_tts/wayu" / split
        ).resolve()
        assert source_path.is_relative_to(expected_source_root)
        assert hashlib.sha256(source_path.read_bytes()).hexdigest() == row["audio_file_sha256"]
        audio, source = read_and_resample(source_path)
        assert source["source_sample_rate"] == int(row["sample_rate"])
        assert source["source_channels"] == int(row["channels"])
        assert abs(source["source_duration_s"] - float(row["duration_s"])) <= 1 / source["source_sample_rate"] + 1e-8

        content = encode_float32_wav(audio)
        output_path = PILOT_DIR / "canonical/clean16k" / split / label / f"{row['sample_id']}.wav"
        assert output_path.resolve().is_relative_to((PILOT_DIR / "canonical/clean16k").resolve())
        prepared_files[output_path] = content
        duration = len(audio) / TARGET_SAMPLE_RATE
        assert abs(duration - source["source_duration_s"]) <= 1 / TARGET_SAMPLE_RATE + 1e-8
        item = row.to_dict()
        item.update({
            "source_audio_path": row["audio_path"],
            "source_audio_file_sha256": row["audio_file_sha256"],
            **source,
            "audio_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
            "sample_rate": TARGET_SAMPLE_RATE, "channels": 1,
            "frames": int(audio.size), "duration_s": duration,
            "peak": float(np.abs(audio).max()),
            "audio_file_sha256": hashlib.sha256(content).hexdigest(),
            "waveform_sha256": hashlib.sha256(audio.astype("<f4", copy=False).tobytes()).hexdigest(),
            "audio_policy": CANONICAL_POLICY,
        })
        canonical_rows.append(item)
        if index % 25 == 0:
            print(f"เตรียม {index}/200 คลิป", flush=True)

df_canonical = pd.DataFrame(canonical_rows)
assert len(df_canonical) == 200
assert df_canonical["sample_id"].is_unique
assert df_canonical["audio_file_sha256"].is_unique
assert df_canonical["waveform_sha256"].is_unique, "หลังแปลงเสียงมี waveform ซ้ำ ต้องตรวจ"
for column in ["source_sample_id", "source_text_key", "audio_file_sha256", "waveform_sha256"]:
    assert set(df_canonical[df_canonical["split"].eq("train")][column]).isdisjoint(
        set(df_canonical[df_canonical["split"].eq("dev")][column])
    ), column
assert df_canonical["sample_rate"].eq(16_000).all() and df_canonical["channels"].eq(1).all()
print("ตรวจครบ 200; ยังเก็บความยาวเต็ม ไม่ crop/pad")

## 2. บันทึก derived audio / manifests / รายงานเทคนิค

ไฟล์ใหม่อยู่ `canonical/clean16k/<split>/<label>/` และ manifest `wayu_pilot_<split>_clean16k.csv` มี `source_audio_path`/`source_audio_file_sha256` สำหรับย้อนกลับไปตรวจต้นทาง

รายงานเก็บ policy/software/code/source/output hashes และผลตรวจเทคนิค ไม่เก็บความคิดเห็นส่วนตัว หาก artifact version เดิมมีเนื้อหาต่างกันจะหยุดก่อนเขียน ไม่ลบหรือสร้าง backup

In [ ]:
canonical_tables = {}
manifest_hashes = {}
for split in ["train", "dev"]:
    frame = df_canonical[df_canonical["split"].eq(split)].copy()
    canonical_tables[split] = frame
    filename = f"wayu_pilot_{split}_clean16k.csv"
    content = frame.to_csv(index=False, lineterminator="\n", float_format="%.10g").encode("utf-8")
    prepared_files[MANIFEST_DIR / filename] = content
    manifest_hashes[filename] = hashlib.sha256(content).hexdigest()

helper_paths = [
    PROJECT_ROOT / "src/thai_spoof/cvtts/audio.py",
    PROJECT_ROOT / "src/thai_spoof/cvtts/artifacts.py",
]
report = {
    "experiment": EXPERIMENT, "scope": "shared_audio_preparation_for_feasibility_pilot_only",
    "audio_policy": CANONICAL_POLICY,
    "settings": {
        "format": "WAV", "subtype": "FLOAT", "sample_rate": 16_000, "channels": 1,
        "downmix": "arithmetic_mean", "resampling": "scipy.signal.resample_poly",
        "amplitude_normalization": "none", "clip_amplitude": False,
        "duration": "keep_full_clip", "crop_pad_noise_telephone": False,
    },
    "input_report": INPUT_REPORT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "input_report_sha256": hashlib.sha256(INPUT_REPORT_PATH.read_bytes()).hexdigest(),
    "source_manifest_sha256": source_manifest_hashes,
    "manifest_sha256": manifest_hashes,
    "python": platform.python_version(),
    "software": {p: metadata.version(p) for p in ["numpy", "scipy", "soundfile", "pandas"]},
    "preparation_code_sha256": {
        p.relative_to(PROJECT_ROOT).as_posix(): hashlib.sha256(p.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
        for p in helper_paths
    },
    "counts": {"train": {"bonafide": 80, "spoof": 80}, "dev": {"bonafide": 20, "spoof": 20}},
    "technical_checks": {
        "source_hashes_match": True, "finite_nonempty_nonzero": True,
        "float32_wav_roundtrip": True, "duration_error_at_most_one_target_sample": True,
        "duplicate_output_file_hashes": 0, "duplicate_output_waveform_hashes": 0,
        "cross_split_source_or_output_overlap": 0,
    },
    "source_codec_or_bandwidth_effects_are_not_removed": True,
    "not_a_human_speech_quality_verdict_or_detector_evaluation": True,
    "training_performed": False,
}
REPORT_PATH = PILOT_DIR / "qc/canonical_audio" / f"{EXPERIMENT}.json"
prepared_files[REPORT_PATH] = (json.dumps(report, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
created = write_immutable_artifacts(prepared_files)

# อ่านไฟล์ที่บันทึกจริงกลับทุกคลิป ไม่ตรวจเฉพาะไฟล์ใน memory
for row in canonical_rows:
    path = PROJECT_ROOT / row["audio_path"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["audio_file_sha256"]
    decoded, sr = sf.read(path, dtype="float32")
    assert sr == 16_000 and sf.info(path).subtype == "FLOAT"
    assert decoded.ndim == 1 and decoded.size == row["frames"] and np.isfinite(decoded).all()
    assert hashlib.sha256(decoded.astype("<f4", copy=False).tobytes()).hexdigest() == row["waveform_sha256"]
for filename, digest in source_manifest_hashes.items():
    assert hashlib.sha256((MANIFEST_DIR / filename).read_bytes()).hexdigest() == digest

print("ไฟล์ใหม่ที่สร้างรอบนี้:", created, "(200 WAV + 2 manifests + 1 report ในรอบแรก)")
print("เวลา:", round(time.perf_counter() - started, 2), "วินาที")
display(df_canonical.groupby(["split", "label"]).size().rename("clips").to_frame())
display(df_canonical.groupby(["split", "label"])["duration_s"].agg(["min", "median", "max"]))
display(df_canonical[["sample_id", "label", "sample_rate", "channels", "duration_s"]].head())
print("ยังไม่ train และยังไม่ได้ฟังครบทุกคลิปใหม่")

## 3. ฟังเทียบไฟล์ต้นทางกับไฟล์ 16 kHz

ตัวอย่าง Train ใช้ข้อความ **“ฉันไม่ได้คุยโม้”** ที่เคยฟังแล้ว มี 4 players: เสียงคนก่อน/หลัง และ Wayu ก่อน/หลัง การแปลง sample rate อาจเปลี่ยนรายละเอียดเสียง แต่ไม่ควรทำให้คำหายหรือขาดตอน

ฟังเฉพาะการแปลงไฟล์ ไม่เลือกค่าตัวสร้างหรือ detector จาก Dev หลังขั้นนี้จึงค่อยกำหนด window ใน DataLoader และเพิ่ม training loop AASIST เพื่อทดลอง Clean 1 epoch ยังไม่มีคำสั่ง train สำเร็จรูปจาก notebook นี้

In [ ]:
pair = canonical_tables["train"][canonical_tables["train"]["source_sample_id"].eq("common_voice_th_25669007")]
assert len(pair) == 2 and set(pair["label"]) == {"bonafide", "spoof"}
for label, title in [("bonafide", "เสียงคน"), ("spoof", "Wayu")]:
    row = pair[pair["label"].eq(label)].iloc[0]
    source_path = PROJECT_ROOT / row["source_audio_path"]
    output_path = PROJECT_ROOT / row["audio_path"]
    assert hashlib.sha256(source_path.read_bytes()).hexdigest() == row["source_audio_file_sha256"]
    assert hashlib.sha256(output_path.read_bytes()).hexdigest() == row["audio_file_sha256"]
    display(Markdown(f"### {title}: {row['sentence']}"))
    display(Markdown(f"ต้นทาง — {row['source_sample_rate']} Hz"))
    display(Audio(filename=str(source_path), normalize=False))
    display(Markdown("ไฟล์ใหม่ — mono 16 kHz WAV FLOAT ความยาวเต็ม"))
    display(Audio(filename=str(output_path), normalize=False))
print("ช่วยเช็กว่าเสียงหลังแปลงยังคำครบ ไม่มีการขาดตอนจากการแปลง")
print("ถัดไป: training DataLoader/window และลอง AASIST Clean 1 epoch ไม่ใช่ Final Test")